# RF-DETR Keypoint — Dog Pose Fine-tuning

Fine-tunes **RF-DETR Keypoint Nano** on the Ultralytics dog-pose dataset (8,476 images, 24 keypoints).  
The trained model is the perception backbone for the `animal2robot` gait-extraction pipeline.

**Pipeline:**
```
dog video → RF-DETR Keypoint → per-frame keypoints → gait representation → future GO2 motion prior
```

**Runtime:** GPU required. T4 (Colab free tier) works with the settings below.

## 1. Environment

> **Before running:** Go to *Runtime → Change runtime type → GPU → A100* for best results.

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q "rfdetr[train,visual]" roboflow supervision ultralytics

In [ ]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
import supervision as sv
from IPython.display import Image, display

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Mount Google Drive to persist the trained checkpoint across sessions
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/animal2robot")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint will be saved to: {DRIVE_DIR}")

## 2. Dataset

Using the **Ultralytics dog-pose dataset**: 8,476 images with 24 keypoints per dog in YOLO format.  
Keypoints cover nose, eyes, ears, spine, all 4 limbs with shoulder/elbow/paw or hip/knee/paw.

The 24 keypoints:
```
 0: nose            12: right_front_paw
 1: left_eye        13: right_front_wrist
 2: right_eye       14: right_front_elbow
 3: left_ear        15: right_front_shoulder
 4: right_ear       16: left_back_paw
 5: throat          17: left_back_hock
 6: withers         18: left_back_knee
 7: tail_base       19: left_back_hip
 8: left_front_paw  20: right_back_paw
 9: left_front_wrist 21: right_back_hock
10: left_front_elbow 22: right_back_knee
11: left_front_shoulder 23: right_back_hip
```

In [ ]:
import urllib.request
import zipfile

DATASET_ROOT = Path("datasets")
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

# The zip extracts directly into DATASET_ROOT (no subdirectory)
DATASET_DIR = DATASET_ROOT

if not (DATASET_DIR / "images" / "train").exists():
    print("Downloading Ultralytics dog-pose dataset (~200 MB)...")
    urllib.request.urlretrieve(
        "https://ultralytics.com/assets/dog-pose.zip",
        DATASET_ROOT / "dog-pose.zip"
    )
    with zipfile.ZipFile(DATASET_ROOT / "dog-pose.zip", "r") as z:
        z.extractall(DATASET_ROOT)
    print("Done.")
else:
    print("Dataset already present.")

for split in ["train", "val"]:
    imgs   = list((DATASET_DIR / "images" / split).glob("*.jpg"))
    labels = list((DATASET_DIR / "labels" / split).glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(labels)} label files")

print(f"\nDATASET_DIR = {DATASET_DIR.resolve()}")

In [ ]:
# Restructure dataset to match rfdetr's expected layout:
#   data.yaml
#   train/images/*.jpg
#   train/labels/*.txt
#   val/images/*.jpg
#   val/labels/*.txt

import shutil

DATASET_DIR = DATASET_ROOT  # already set above

def restructure_dataset(root: Path):
    for split in ["train", "val"]:
        src_images = root / "images" / split
        src_labels = root / "labels" / split
        dst_images = root / split / "images"
        dst_labels = root / split / "labels"

        if src_images.exists() and not dst_images.exists():
            print(f"Moving {src_images} → {dst_images}")
            dst_images.mkdir(parents=True, exist_ok=True)
            for f in src_images.iterdir():
                shutil.move(str(f), str(dst_images / f.name))

        if src_labels.exists() and not dst_labels.exists():
            print(f"Moving {src_labels} → {dst_labels}")
            dst_labels.mkdir(parents=True, exist_ok=True)
            for f in src_labels.iterdir():
                shutil.move(str(f), str(dst_labels / f.name))

    # Rename yaml to data.yaml if needed
    yaml_src = root / "dog-pose.yaml"
    yaml_dst = root / "data.yaml"
    if yaml_src.exists() and not yaml_dst.exists():
        print(f"Renaming {yaml_src.name} → data.yaml")
        shutil.copy(str(yaml_src), str(yaml_dst))

restructure_dataset(DATASET_DIR)

# Verify
for split in ["train", "val"]:
    imgs   = list((DATASET_DIR / split / "images").glob("*.jpg"))
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(labels)} labels")
print(f"data.yaml exists: {(DATASET_DIR / 'data.yaml').exists()}")

In [ ]:
# Inspect the actual directory structure to find where labels landed
import os

print(f"DATASET_DIR = {DATASET_DIR}\n")
for root, dirs, files in os.walk(str(DATASET_DIR)):
    depth = root.replace(str(DATASET_DIR), "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{Path(root).name}/")
    if depth <= 2:
        for fname in sorted(files)[:5]:
            print(f"{indent}  {fname}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

In [ ]:
# Once you see the structure above, update these paths to match
# Common alternatives: images/train2017, train/images, etc.
# Then verify the annotation format

label_files = list(DATASET_DIR.rglob("*.txt"))
label_files = [f for f in label_files if "labels" in str(f)]  # skip data.yaml etc.

if not label_files:
    print("No label files found — check the directory tree above and adjust DATASET_DIR")
else:
    print(f"Found {len(label_files)} label files. Sample: {label_files[0]}")
    with open(label_files[0]) as f:
        first_line = f.readline().strip()
    if first_line:
        values = first_line.split()
        print(f"Values per annotation: {len(values)}")
        print(f"  5 box + {(len(values)-5)//3} keypoints × 3 = {len(values)} total")
    else:
        print("Label file is empty — may be a background image.")

## 3. Training

Fine-tuning **RF-DETR Keypoint Preview** (the only available keypoint variant) from the COCO pre-trained checkpoint.  
Settings are chosen to fit within T4 VRAM (15 GB).

In [ ]:
# Training configuration — A100 settings
EPOCHS      = 50
BATCH_SIZE  = 8
RESOLUTION  = 576
LR          = 1e-4
GRAD_ACCUM  = 1
OUTPUT_DIR  = DRIVE_DIR / "rfdetr_dog_pose"  # saved to Drive, survives session end
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: {(torch.cuda.mem_get_info()[0]/1e9):.1f} GB")
print(f"Checkpoint dir: {OUTPUT_DIR}")

In [ ]:
from rfdetr import RFDETRKeypointPreview

model = RFDETRKeypointPreview()

model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    resolution=RESOLUTION,
    lr=LR,
    grad_accum_steps=GRAD_ACCUM,
    output_dir=str(OUTPUT_DIR),
)

## 4. Evaluation

In [ ]:
# Find best checkpoint
checkpoints = sorted(OUTPUT_DIR.glob("**/*.pth"))
if checkpoints:
    best_ckpt = checkpoints[-1]
    print(f"Best checkpoint: {best_ckpt}")
else:
    print("No checkpoint found — check training output above.")

In [ ]:
# Load trained model for inference
from rfdetr import RFDETRKeypointPreview
inference_model = RFDETRKeypointPreview(pretrain_weights=str(best_ckpt))
print("Model loaded.")

## 5. Inference on a Single Image

In [ ]:
# Pick a validation image for visual inspection
val_images = list((DATASET_DIR / "images" / "val").glob("*.jpg"))
TEST_IMAGE = str(val_images[0])
print(f"Test image: {TEST_IMAGE}")
display(Image(TEST_IMAGE, width=400))

In [ ]:
# Run inference
results = inference_model.predict(TEST_IMAGE, threshold=0.3)
print(f"Detections: {len(results)}")
if results:
    kps = results[0]  # first detection
    print(f"Keypoints returned: {len(kps.keypoints) if hasattr(kps, 'keypoints') else 'N/A'}")
    print(results[0])

## 6. Visualize Skeleton Overlay

In [ ]:
# Dog skeleton edges (index pairs from the 24-keypoint definition)
# Each tuple is (keypoint_a_index, keypoint_b_index)
DOG_SKELETON_EDGES = [
    (0, 5),   # nose → throat
    (5, 6),   # throat → withers
    (6, 7),   # withers → tail_base
    # Front left leg
    (6, 11),  # withers → left_front_shoulder
    (11, 10), # left_front_shoulder → left_front_elbow
    (10, 9),  # left_front_elbow → left_front_wrist
    (9, 8),   # left_front_wrist → left_front_paw
    # Front right leg
    (6, 15),  # withers → right_front_shoulder
    (15, 14), # right_front_shoulder → right_front_elbow
    (14, 13), # right_front_elbow → right_front_wrist
    (13, 12), # right_front_wrist → right_front_paw
    # Rear left leg
    (7, 19),  # tail_base → left_back_hip
    (19, 18), # left_back_hip → left_back_knee
    (18, 17), # left_back_knee → left_back_hock
    (17, 16), # left_back_hock → left_back_paw
    # Rear right leg
    (7, 23),  # tail_base → right_back_hip
    (23, 22), # right_back_hip → right_back_knee
    (22, 21), # right_back_knee → right_back_hock
    (21, 20), # right_back_hock → right_back_paw
]

KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "throat", "withers", "tail_base",
    "left_front_paw", "left_front_wrist", "left_front_elbow", "left_front_shoulder",
    "right_front_paw", "right_front_wrist", "right_front_elbow", "right_front_shoulder",
    "left_back_paw", "left_back_hock", "left_back_knee", "left_back_hip",
    "right_back_paw", "right_back_hock", "right_back_knee", "right_back_hip",
]

# Highlight the 4 paws used for gait extraction
PAW_INDICES = [8, 12, 16, 20]  # LF, RF, LH, RH
print("Paw keypoints:", [KEYPOINT_NAMES[i] for i in PAW_INDICES])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_dog_skeleton(image_path, predictions, conf_threshold=0.3):
    """Draw keypoints and skeleton edges on the image."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)

    for det in predictions:
        # Extract keypoints — adjust attribute access to match rfdetr output
        kps = det.keypoints if hasattr(det, 'keypoints') else []
        if not kps:
            continue

        coords = [(kp.x, kp.y, kp.confidence) for kp in kps]

        # Draw skeleton edges
        for (a, b) in DOG_SKELETON_EDGES:
            if a >= len(coords) or b >= len(coords):
                continue
            xa, ya, ca = coords[a]
            xb, yb, cb = coords[b]
            if ca >= conf_threshold and cb >= conf_threshold:
                ax.plot([xa, xb], [ya, yb], '-', color='lime', linewidth=1.5, alpha=0.8)

        # Draw keypoints
        for i, (x, y, conf) in enumerate(coords):
            if conf < conf_threshold:
                continue
            color = 'red' if i in PAW_INDICES else 'dodgerblue'
            size  = 80 if i in PAW_INDICES else 40
            ax.scatter(x, y, s=size, c=color, zorder=5)

    # Legend
    ax.legend(handles=[
        mpatches.Patch(color='red',       label='Paw (gait signal)'),
        mpatches.Patch(color='dodgerblue', label='Other keypoint'),
    ], loc='upper right')
    ax.axis('off')
    ax.set_title('RF-DETR Keypoint — Dog Skeleton Overlay')
    plt.tight_layout()
    plt.savefig('outputs/skeleton_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_dog_skeleton(TEST_IMAGE, results)

## 7. Inference on Video

Upload a short side-view dog walking/trotting video, then run frame-by-frame inference.

In [ ]:
# Upload a video file
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {VIDEO_PATH}")

In [ ]:
import pandas as pd
from tqdm import tqdm

def run_video_inference(video_path, model, conf_threshold=0.3):
    """
    Extract per-frame keypoints from a video.
    Returns a DataFrame with columns: frame, timestamp, keypoint, x, y, confidence
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Video: {total_frames} frames @ {fps:.1f} FPS, {width}x{height}")

    records = []
    frame_id = 0

    with tqdm(total=total_frames, desc="Processing frames") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            timestamp = frame_id / fps

            # Save frame temporarily for inference
            tmp_path = "/tmp/frame.jpg"
            cv2.imwrite(tmp_path, frame)
            predictions = model.predict(tmp_path, threshold=conf_threshold)

            # Take highest-confidence detection (one dog assumption)
            if predictions:
                det = predictions[0]
                kps = det.keypoints if hasattr(det, 'keypoints') else []
                for i, kp in enumerate(kps):
                    records.append({
                        "frame":      frame_id,
                        "timestamp":  round(timestamp, 4),
                        "keypoint":   KEYPOINT_NAMES[i] if i < len(KEYPOINT_NAMES) else f"kp_{i}",
                        "x":          kp.x,
                        "y":          kp.y,
                        "confidence": round(kp.confidence, 4),
                    })

            frame_id += 1
            pbar.update(1)

    cap.release()

    df = pd.DataFrame(records)
    meta = {"fps": fps, "width": width, "height": height, "total_frames": total_frames}
    return df, meta


df_keypoints, video_meta = run_video_inference(VIDEO_PATH, inference_model)
print(f"\nExtracted {len(df_keypoints)} keypoint observations across {df_keypoints['frame'].nunique()} frames")
df_keypoints.head(10)

In [ ]:
# Save predictions to CSV
os.makedirs("outputs", exist_ok=True)
df_keypoints.to_csv("outputs/keypoints.csv", index=False)
with open("outputs/video_meta.json", "w") as f:
    json.dump(video_meta, f, indent=2)
print("Saved: outputs/keypoints.csv and outputs/video_meta.json")

## 8. Quick Detection Quality Check

In [ ]:
import matplotlib.pyplot as plt

# Detection rate per keypoint
total_frames = video_meta["total_frames"]
paw_names = [KEYPOINT_NAMES[i] for i in PAW_INDICES]

detection_rates = (
    df_keypoints[df_keypoints["confidence"] >= 0.3]
    .groupby("keypoint")["frame"]
    .nunique()
    / total_frames
    * 100
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["red" if k in paw_names else "steelblue" for k in detection_rates.index]
ax.bar(detection_rates.index, detection_rates.values, color=colors)
ax.axhline(80, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
ax.set_ylabel("Detection rate (%)")
ax.set_title("Keypoint Detection Rate Across Video Frames")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig("outputs/detection_rates.png", dpi=150)
plt.show()

print("\nPaw detection rates:")
for name in paw_names:
    rate = detection_rates.get(name, 0)
    print(f"  {name}: {rate:.1f}%")

## Next Steps

The `outputs/keypoints.csv` file is the input to the **gait extraction pipeline** (`src/gait.py`):

```
outputs/keypoints.csv
    ↓
Confidence filtering + interpolation + Savitzky-Golay smoothing
    ↓
Translation + scale normalization (body_center = midpoint(withers, tail_base))
    ↓
Four-paw vertical trajectories
    ↓
Stride period (FFT + autocorrelation)
    ↓
Inter-limb phase (cross-correlation relative to left_front_paw)
    ↓
Structured gait JSON → future Unitree GO2 motion prior
```